# SETUP NOTES
This requires:
* a `databricks` schema in `metadata_{env}`
* supporting views defined in __Billing Supporting Views__ notebook

# Workspaces
Currenlty in our environment there are no repeated rows in workspaces_latest.  
There are, however, billing usage rows without defined workspaces

In [0]:
select    *
from      system.access.workspaces_latest

# List Prices
There can be multiple of a given combination of sku, cloud, currency_code, and usage_unit

These differentiate on start time; the value changes over time, presumably with new contracts signed, but I find no breaks in the sequence.

Thus, a given cost will tie to a list price effective at the time the expense was incurred.

In [0]:
select    count(*) num_rows
      ,   lp.sku_name, lp.cloud, lp.currency_code, lp.usage_unit --, lp.pricing
      ,   lp.price_start_time
from      system.billing.list_prices lp
group by  lp.sku_name, lp.cloud, lp.currency_code, lp.usage_unit
      ,   lp.price_start_time
having    count(*) > 1

In [0]:
select      p.price_end_time, lp.price_start_time, lp.price_end_time
        ,   lp.sku_name, lp.cloud, lp.currency_code, lp.usage_unit, lp.pricing.default, lp.pricing.effective_list.default
        ,   lp.pricing.default - lp.pricing.effective_list.default effective_discount
from        system.billing.list_prices lp
left join        system.billing.list_prices p on p.sku_name = lp.sku_name and p.cloud = lp.cloud and p.price_end_time = lp.price_start_time
where       lp.sku_name in ('PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE_US_EAST', 'PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE_US_EAST_2', 'PREMIUM_JOBS_COMPUTE')
order by    lp.sku_name, lp.price_start_time

### Notes & Deferred Study
How is an expense handled when it starts under one effective list price and completes under another?  This may be a wild goose, but is theoretically possible.

# Custom Tags
These are the tags passed through from Azure resource definitions.

They make a good way of segmenting expense to account codes, teams, departments, etc. _as long as the tags are setup correctly and used consistently_

In [0]:
select    custom_tags.appName, custom_tags.environment, *
from      system.billing.usage
where     usage_date > '2025-07-20'
limit     50

# Jobs
Jobs can have multiple rows to account for creation, rename, deletion, recreation, etc.  

Can simply use most recent name of job or get fancy and go with the name in place when the job was run.  

(I currently see little use in the latter)

In [0]:
select    *
from      system.lakeflow.jobs 
where     job_id = 846236281836643

## Job & Task Runs
Jobs invoked by API (Azure Data Factory, etc.) do NOT get rows in `system.lakeview.jobs` but do show up in `system.lakeview.job_run_timeline`

We can get the `run_name` from `job_run_timeline` and attribute cost to jobs invoked via API (such as Azure Data Factory invoked jobs)

The run_name ends with the runid guid for the activity in the ADF pipeline;  
if we ignore the last 37 characters we can group all jobs run from a given pipeline activity.

In [0]:
select    j.job_id, u.usage_metadata.job_id, u.*
from      system.billing.usage u
left join system.lakeflow.jobs j on j.job_id = u.usage_metadata.job_id
where     u.usage_metadata.job_id in (261906092054121, 515409413376377)

In [0]:
/* ADF invoked jobs get a run name with pattern ADF_<factory>_<pipeline>_<activity>_<runid guid>

-- Carve off the _runid (last 37 characters) and we get something we can group by.
-- This is a split method, assuming a 5 element array...
select    replace(run_name, split(run_name, '_')[4], '') caller
from      system.lakeflow.job_run_timeline 
where     job_id in (261906092054121, 515409413376377) 
order by  job_id, period_start_time desc

*/

-- However, not everything is called by ADF, and some that are have a different number of elements
-- In our environment, almost everything is either ADF_... or NULL
select    left(run_name, len(run_name) - 37) caller
      ,   count(*) num_runs
from      system.lakeflow.job_run_timeline 
where     run_name like 'ADF%'
group by  left(run_name, len(run_name) - 37)

In [0]:
select      *
from        system.lakeflow.job_task_run_timeline 
where       job_id in (261906092054121, 515409413376377) 
order by    job_id, period_start_time desc

# Pipelines (DLT and?)
As with jobs, pipelines get a new row each time they are modified.

Use the most recent name, type, etc. for now.

In [0]:
/*
select      pl.pipeline_id, count(*) num_rows, min(change_time) earliest, max(change_time) latest
from        system.lakeflow.pipelines pl
group by    pl.pipeline_id
having      count(*) > 1
limit 10
*/

select      *
from        system.lakeflow.pipelines
where       pipeline_id in ('a17971c8-37d0-4882-8922-e43bd14499d8', 'cf90614e-1617-4eec-82d2-4205e5d28cd4')
order by    pipeline_id, change_time

In [0]:
select      pl.pipeline_type, count(*)
from        system.lakeflow.pipelines pl
group by    pl.pipeline_type

# Usage & Activity 
### Job, Pipeline, DLT, etc.
Usage is always tied to a workspace and has access to its custom tags.

Usage _might_ be tied to jobs, clusters, DLT, warehouses, notebooks or apps (or some combination of these)

In [0]:
select      u.sku_name, u.usage_type, u.usage_unit
        ,   count(*) num_rows

        ,   sum(case when u.workspace_id is not null then 1 else 0 end) has_workspace
        ,   sum(case when u.usage_metadata.cluster_id is not null then 1 else 0 end) has_cluster
        ,   sum(case when u.usage_metadata.job_id is not null then 1 else 0 end) has_job
        ,   sum(case when u.usage_metadata.warehouse_id is not null then 1 else 0 end) has_warehouse
        ,   sum(case when u.usage_metadata.dlt_pipeline_id is not null then 1 else 0 end) has_dlt_pipeline
        ,   sum(case when u.usage_metadata.dlt_maintenance_id is not null then 1 else 0 end) has_dlt_maintenance
        ,   sum(case when u.usage_metadata.dlt_update_id is not null then 1 else 0 end) has_dlt_update
        ,   sum(case when u.usage_metadata.notebook_id is not null then 1 else 0 end) has_notebook
        ,   sum(case when u.usage_metadata.app_id is not null then 1 else 0 end) has_app

        ,   sum(case when u.usage_metadata.cluster_id is null
                      and u.usage_metadata.job_id is null
                      and u.usage_metadata.warehouse_id is null
                      and u.usage_metadata.dlt_pipeline_id is null
                      and u.usage_metadata.dlt_maintenance_id is null
                      and u.usage_metadata.dlt_update_id is null 
                      and u.usage_metadata.notebook_id is null
                      and u.usage_metadata.app_id is null 
                      then 1 else 0 end) none_of_these
from        system.billing.usage u
group by    u.sku_name, u.usage_type, u.usage_unit
order by    u.sku_name, u.usage_type, u.usage_unit

In [0]:
select    usage_metadata
from      system.billing.usage u
where     u.usage_metadata.job_id is null
      and u.usage_metadata.cluster_id is null
      and u.usage_metadata.warehouse_id is null
      and u.usage_metadata.dlt_pipeline_id is null
      and u.usage_metadata.dlt_maintenance_id is null
      and u.usage_metadata.dlt_update_id is null 
      and u.usage_metadata.notebook_id is null
      and u.usage_metadata.app_id is null   
      and u.usage_date > '2025-07-01'
limit 50

### Combinations of job, warehouse, dlt, etc.

* Notebooks, apps, and warehouses intersect with nothing
* jobs, clusters and dlt_pipelines may stand alone or: 
  * jobs may intersect with clusters
  * dlt_pipelines may intersect with dlt_maintenance, dlt_update, or clusters (never more than one at a time)
  * dlt_maintenance and dlt_update are always tied to a dlt_piepline
* usage_metadata may be null for all these

In [0]:
select    distinct 
          case when u.usage_metadata.job_id is not null then 1 else 0 end has_job
      ,   case when u.usage_metadata.cluster_id is not null then 1 else 0 end has_cluster
      ,   case when u.usage_metadata.warehouse_id is not null then 1 else 0 end has_warehouse
      ,   case when u.usage_metadata.dlt_pipeline_id is not null then 1 else 0 end has_dlt_pipeline
      ,   case when u.usage_metadata.dlt_maintenance_id is not null then 1 else 0 end has_dlt_maintenance
      ,   case when u.usage_metadata.dlt_update_id is not null  then 1 else 0 end has_dlt_update
      ,   case when u.usage_metadata.notebook_id is not null then 1 else 0 end has_notebook
      ,   case when u.usage_metadata.app_id is not null then 1 else 0 end has_app
from      system.billing.usage u
where     1 = 1

### Details
* Clusters refer to `system.compute.clusters`
* Warehouses refer to `system.compute.warehouses`
* Jobs refer to `system.lakeflow.jobs`
* Apps are defined within the `usage_metadata` - Id and Name  
* Notebooks are defined within the `usage_metadata` - Id and Path  
* DLT Pipelines are defined in two places:
  * `usage_metadata` - UC catalog, schema, and table
  * `system.lakeflow.pipelines` - pipeline_type, history

In [0]:
select      u.workspace_id, pl.name, u.sku_name, u.billing_origin_product, u.usage_type
        ,   u.usage_metadata.dlt_pipeline_id, u.usage_metadata.dlt_maintenance_id, u.usage_metadata.dlt_update_id
        ,   u.usage_metadata.uc_table_catalog, u.usage_metadata.uc_table_schema, u.usage_metadata.uc_table_name
        ,   u.usage_date, u.usage_quantity
        ,   pl.pipeline_type, pl.change_time
from        system.billing.usage u
join        system.lakeflow.pipelines pl on pl.pipeline_id = u.usage_metadata.dlt_pipeline_id
where       u.usage_metadata.dlt_pipeline_id in ('eed7f771-58a1-480d-8de8-2aad36c2f92f')
;

In [0]:
with rn as (
    select    'Job' group_type, u.usage_metadata, row_number() over (order by u.usage_date desc) row_num
    from      system.billing.usage u
    where     u.usage_metadata.job_id is not null
    union
    select    'App' group_type, u.usage_metadata, row_number() over (order by u.usage_date desc) row_num
    from      system.billing.usage u
    where     u.usage_metadata.app_id is not null
    union
    select    'Notebook' group_type, u.usage_metadata, row_number() over (order by u.usage_date desc) row_num
    from      system.billing.usage u
    where     u.usage_metadata.notebook_id is not null
    union
    select    'Warehouse' group_type, u.usage_metadata, row_number() over (order by u.usage_date desc) row_num
    from      system.billing.usage u
    where     u.usage_metadata.warehouse_id is not null
    union
    select    'DLT Pipeline' group_type, u.usage_metadata, row_number() over (order by u.usage_date desc) row_num
    from      system.billing.usage u
    where     u.usage_metadata.dlt_pipeline_id is not null
)

select * from rn where row_num = 1

# Billing_Usage
Now put it all together

In [0]:
select      workspace_id, count(*) num_rows, min(usage_date) earliest, max(usage_date) latest
from        system.billing.usage
where       workspace_id not in (select workspace_id from system.access.workspaces_latest )
group by    workspace_id

In [0]:
select    s.workspace_name, s.usage_unit, s.currency_code
      ,   round(m0, 2) as m0, round(m1, 2) as m1, round(m2, 2) as m2, round(m3, 2) as m2, round(m4, 2) as m4
      ,   round(m5, 2) as m5, round(m6, 2) as m6, round(m7, 2) as m7, round(m8, 2) as m8, round(m9, 2) as m9
      ,   round(m1, 2) as m10, round(m1, 2) as m11, round(m1, 2) as m12
from (
      select          w.workspace_name, cal.months_ago, u.usage_unit, lp.currency_code
              ,       u.usage_quantity * lp.pricing.effective_list.default usage_cost
      from            system.billing.usage u
      join            silver_dev.edm.dim_calendar_rolling_13_months cal on cal.date_ = u.usage_date
      join            system.billing.list_prices lp on lp.sku_name = u.sku_name and u.usage_date >= lp.price_start_time and (u.usage_date <= lp.price_end_time or lp.price_end_time is null)
      left join       system.access.workspaces_latest w on w.workspace_id = u.workspace_id
) s
pivot (sum(s.usage_cost) for months_ago in (0 as m0, 1 as m1, 2 as m2, 3 as m3, 4 as m4, 5 as m5, 6 as m6, 7 as m7, 8 as m8, 9 as m9, 10 as m10, 11 as m11, 12 as m12))
order by        s.workspace_name, s.usage_unit, s.currency_code


In [0]:
select    s.workspace_name, s.usage_unit, s.currency_code, s.sku_name, s.billing_origin_product, s.usage_type
      ,   round(m0, 2) as m0, round(m1, 2) as m1, round(m2, 2) as m2, round(m3, 2) as m2, round(m4, 2) as m4
      ,   round(m5, 2) as m5, round(m6, 2) as m6, round(m7, 2) as m7, round(m8, 2) as m8, round(m9, 2) as m9
      ,   round(m1, 2) as m10, round(m1, 2) as m11, round(m1, 2) as m12
from (
    select          w.workspace_name, cal.months_ago, u.sku_name, u.billing_origin_product, u.usage_type, u.usage_unit, lp.currency_code
            ,       u.usage_quantity * lp.pricing.effective_list.default usage_cost
    from            system.billing.usage u
    join            silver_dev.edm.dim_calendar_rolling_13_months cal on cal.date_ = u.usage_date
    join            system.billing.list_prices lp on lp.sku_name = u.sku_name and u.usage_date >= lp.price_start_time and (u.usage_date <= lp.price_end_time or lp.price_end_time is null)
    left join       system.access.workspaces_latest w on w.workspace_id = u.workspace_id
    where           1 = 1
            and     w.workspace_name in ('dbw-dev-entdatalakehouse', 'dbw-prod-entdatalakehouse')
) s
pivot (sum(s.usage_cost) for months_ago in (0 as m0, 1 as m1, 2 as m2, 3 as m3, 4 as m4, 5 as m5, 6 as m6, 7 as m7, 8 as m8, 9 as m9, 10 as m10, 11 as m11, 12 as m12))
order by        s.workspace_name, s.usage_unit, s.currency_code


In [0]:
select    s.workspace_name, s.usage_unit, s.currency_code, s.sku_name, s.billing_origin_product, s.usage_type, s.activity_type, s.activity_detail
      ,   round(m0, 2) as m0, round(m1, 2) as m1, round(m2, 2) as m2, round(m3, 2) as m2, round(m4, 2) as m4
      ,   round(m5, 2) as m5, round(m6, 2) as m6, round(m7, 2) as m7, round(m8, 2) as m8, round(m9, 2) as m9
      ,   round(m1, 2) as m10, round(m1, 2) as m11, round(m1, 2) as m12
from (
        select          w.workspace_name, cal.months_ago, u.usage_unit, lp.currency_code
                ,       u.sku_name, u.billing_origin_product, u.usage_type
                ,       case 
                        when j.name is not null then 'Job'
                        when wh.warehouse_name is not null then 'Warehouse'
                        when pl.name is not null then 'DLT Pipeline'
                        when u.usage_metadata.notebook_id is not null then 'Notebook'
                        when u.usage_metadata.app_id is not null then 'App'
                        end activity_type
                ,       coalesce(
                                wh.warehouse_name || ' (' || wh.warehouse_type || ', ' || wh.warehouse_channel || ', ' || u.usage_metadata.warehouse_id || ')'
                        ,       pl.name || ' (' 
                                || coalesce(u.usage_metadata.uc_table_catalog || '.' || u.usage_metadata.uc_table_schema || '.' || u.usage_metadata.uc_table_name || ', ', '') 
                                || pl.pipeline_type || ', ' || u.usage_metadata.dlt_pipeline_id || ')'
                        ,       u.usage_metadata.notebook_path || ' (' || u.usage_metadata.notebook_id || ')'
                        ,       u.usage_metadata.app_name || ' (' || u.usage_metadata.app_id || ')'
                        ,       coalesce(j.name, 'Dynamic (ADF, etc.)')
                        ) activity_detail
                ,       u.usage_quantity * lp.pricing.effective_list.default usage_cost
        from            system.billing.usage u
        join            silver_dev.edm.dim_calendar_rolling_13_months cal on cal.date_ = u.usage_date
        join            system.billing.list_prices lp on        lp.sku_name = u.sku_name 
                                                        and     u.usage_date >= lp.price_start_time 
                                                        and     (u.usage_date <= lp.price_end_time or lp.price_end_time is null)
        left join       system.access.workspaces_latest w on w.workspace_id = u.workspace_id

        left join       metadata_dev.databricks.warehouses_current wh on wh.warehouse_id = u.usage_metadata.warehouse_id
        left join       metadata_dev.databricks.jobs_current j on j.job_id = u.usage_metadata.job_id
        left join       metadata_dev.databricks.pipelines_current pl on pl.pipeline_id = u.usage_metadata.dlt_pipeline_id

        where           1 = 1
                and     w.workspace_name in ('dbw-dev-entdatalakehouse', 'dbw-qa-entdatalakehouse', 'dbw-prod-entdatalakehouse')
) s
pivot (sum(s.usage_cost) for months_ago in (0 as m0, 1 as m1, 2 as m2, 3 as m3, 4 as m4, 5 as m5, 6 as m6, 7 as m7, 8 as m8, 9 as m9, 10 as m10, 11 as m11, 12 as m12))
order by        s.workspace_name, s.usage_unit, s.currency_code


### Notes & Deferred Study
At time of investigation all records in `billing.usage` are `record_type` = 'ORIGINAL'

# What I Want to Do
* Cost by job name / notebook path / app name
* Cost by type by workspace (rollup all jobs, all apps, etc.)
* Cost trends - this month compared to prior month / quarter / year / same month last year


# Hidden Around Corners - Resolving Lump Costs
Are some of the unattributed costs explained in `system.storage.predictive_optimization_operations_history`?  There's a `usage_unit` and `usage_quantity` there...

# Miscellany

In [0]:
%python
from dbruntime.databricks_repl_context import get_context
get_context().workspaceId